# 🤖 AI Document Intelligence & Agent System
### HackBaroda 2026 — Full Upgraded System

## What this system does:
| Agent | Role |
|---|---|
| **Orchestrator** | Decides which agents to call and in what order |
| **Extractor Agent** | Reads PDF / Image / Excel → clean text + structured data |
| **Verifier Agent** | All 5 fraud detection layers (ELA, metadata, signature, QR, text) |
| **Analyst Agent** | AI summary, entity extraction, insights, Q&A via Claude API |
| **Reporter Agent** | Builds final structured report (JSON + readable text) |

## + RAG Pipeline:
- Chunks document → creates embeddings → stores in FAISS
- Lets you **ask questions** about the document and get AI answers

> **Supports:** PDF, PNG, JPG, JPEG, Excel (.xlsx, .csv)

## 📦 Step 1 — Install All Dependencies

In [ ]:
!pip install -q pymupdf pillow opencv-python-headless pytesseract pyzbar \
             pyhanko gradio anthropic faiss-cpu sentence-transformers \
             pandas openpyxl langchain langchain-community tiktoken numpy matplotlib
!apt-get install -q tesseract-ocr libzbar0 poppler-utils
print('✅ All dependencies installed!')

## 🔑 Step 2 — Set Your Claude API Key

In [ ]:
import os

# ── Option A: Paste your key directly (for testing only)
CLAUDE_API_KEY = 'your-claude-api-key-here'   # ← Replace with real key

# ── Option B: Use Colab Secrets (recommended — more secure)
# from google.colab import userdata
# CLAUDE_API_KEY = userdata.get('ANTHROPIC_API_KEY')

os.environ['ANTHROPIC_API_KEY'] = CLAUDE_API_KEY

# ── If you don't have a Claude API key yet:
# Get one free at: https://console.anthropic.com
# Free tier gives enough credits to test

# ── OFFLINE MODE (no API key needed — uses rule-based analysis only)
OFFLINE_MODE = False   # Set True to run without Claude API

print('✅ API key configured!' if CLAUDE_API_KEY != 'your-claude-api-key-here' else '⚠️ Using offline mode — set your API key above')
if CLAUDE_API_KEY == 'your-claude-api-key-here':
    OFFLINE_MODE = True
    print('   Offline mode enabled — all detection layers work, AI analysis will be skipped')

## 🧰 Step 3 — All Imports & Base Setup

In [ ]:
import fitz
import cv2
import numpy as np
import pytesseract
import pandas as pd
import json
import io
import re
import tempfile
import time
import matplotlib.pyplot as plt
from PIL import Image
from datetime import datetime
from pyzbar.pyzbar import decode as qr_decode
from dataclasses import dataclass, field
from typing import Optional

# Claude API
try:
    import anthropic
    claude_client = anthropic.Anthropic(api_key=CLAUDE_API_KEY)
    CLAUDE_AVAILABLE = not OFFLINE_MODE
except Exception:
    CLAUDE_AVAILABLE = False

# Sentence embeddings for RAG
try:
    from sentence_transformers import SentenceTransformer
    import faiss
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    RAG_AVAILABLE = True
    print('✅ RAG pipeline ready (FAISS + sentence-transformers)')
except Exception as e:
    RAG_AVAILABLE = False
    print(f'⚠️ RAG not available: {e}')

print(f'✅ Claude AI: {"ready" if CLAUDE_AVAILABLE else "offline mode"}')
print('✅ All imports loaded!')

## 🤖 Step 4 — Agent Base Class + Memory

In [ ]:
@dataclass
class AgentMemory:
    """Shared memory across all agents in a session."""
    file_path: str = ''
    file_type: str = ''
    raw_text: str = ''
    structured_data: dict = field(default_factory=dict)
    chunks: list = field(default_factory=list)
    embeddings: object = None
    faiss_index: object = None
    extraction_result: dict = field(default_factory=dict)
    verification_result: dict = field(default_factory=dict)
    analysis_result: dict = field(default_factory=dict)
    final_report: dict = field(default_factory=dict)
    agent_log: list = field(default_factory=list)

    def log(self, agent_name, message):
        ts = datetime.now().strftime('%H:%M:%S')
        entry = f'[{ts}] [{agent_name}] {message}'
        self.agent_log.append(entry)
        print(entry)


class BaseAgent:
    """All agents inherit from this."""
    def __init__(self, name: str, memory: AgentMemory):
        self.name = name
        self.memory = memory

    def log(self, msg):
        self.memory.log(self.name, msg)

    def run(self):
        raise NotImplementedError


print('✅ Agent base classes defined!')

## 📄 Step 5 — Extractor Agent (PDF / Image / Excel)

In [ ]:
class ExtractorAgent(BaseAgent):
    """
    Agent 1: Extracts all content from any document type.
    Handles PDF (text + image), Images (OCR), Excel/CSV (structured).
    Stores clean text and structured data in shared memory.
    """

    def run(self):
        self.log('Starting extraction...')
        fp = self.memory.file_path
        ext = fp.lower().split('.')[-1]
        self.memory.file_type = ext

        result = {'status': 'ok', 'method': '', 'pages': 0, 'text_length': 0, 'flags': []}

        if ext == 'pdf':
            result = self._extract_pdf(fp)
        elif ext in ('png', 'jpg', 'jpeg', 'webp', 'tiff'):
            result = self._extract_image(fp)
        elif ext in ('xlsx', 'xls'):
            result = self._extract_excel(fp)
        elif ext == 'csv':
            result = self._extract_csv(fp)
        else:
            result['flags'].append(f'⚠️ Unknown file type: {ext}')

        # Build text chunks for RAG
        self._chunk_text()
        result['text_length'] = len(self.memory.raw_text)
        result['chunk_count'] = len(self.memory.chunks)
        self.memory.extraction_result = result
        self.log(f'Done. {result["text_length"]} chars, {len(self.memory.chunks)} chunks')
        return result

    def _extract_pdf(self, fp):
        result = {'method': 'PyMuPDF + OCR fallback', 'flags': [], 'pages': 0}
        try:
            doc = fitz.open(fp)
            result['pages'] = doc.page_count
            all_text = ''
            for page in doc:
                # Try direct text first (embedded text)
                t = page.get_text()
                if len(t.strip()) < 20:  # Too little text → use OCR
                    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))
                    img = Image.open(io.BytesIO(pix.tobytes('png')))
                    t = pytesseract.image_to_string(img, config='--psm 6')
                    result['flags'].append('ℹ️ Page used OCR (image-based PDF)')
                all_text += t + '\n'
            self.memory.raw_text = all_text.strip()
            doc.close()
            result['flags'].append(f'✅ Extracted {len(all_text)} chars from {result["pages"]} page(s)')
        except Exception as e:
            result['flags'].append(f'❌ PDF extraction error: {e}')
        return result

    def _extract_image(self, fp):
        result = {'method': 'Tesseract OCR', 'flags': [], 'pages': 1}
        try:
            img = Image.open(fp).convert('RGB')
            # Preprocess: upscale small images
            w, h = img.size
            if w < 1000:
                img = img.resize((w*2, h*2), Image.LANCZOS)
                result['flags'].append('ℹ️ Image upscaled for better OCR accuracy')
            # Try multiple OCR configs
            text_default = pytesseract.image_to_string(img, config='--psm 6')
            text_block   = pytesseract.image_to_string(img, config='--psm 3')
            # Use whichever gives more text
            self.memory.raw_text = text_default if len(text_default) >= len(text_block) else text_block
            result['flags'].append(f'✅ OCR extracted {len(self.memory.raw_text)} chars')
        except Exception as e:
            result['flags'].append(f'❌ Image OCR error: {e}')
        return result

    def _extract_excel(self, fp):
        result = {'method': 'Pandas (Excel)', 'flags': [], 'pages': 0}
        try:
            xl = pd.ExcelFile(fp)
            all_text = ''
            summary = {}
            for sheet in xl.sheet_names:
                df = xl.parse(sheet)
                result['pages'] += 1
                summary[sheet] = {
                    'rows': len(df),
                    'cols': len(df.columns),
                    'columns': list(df.columns),
                    'numeric_cols': list(df.select_dtypes(include='number').columns),
                    'null_count': int(df.isnull().sum().sum()),
                    'sample': df.head(3).to_dict(orient='records')
                }
                all_text += f'Sheet: {sheet}\n{df.to_string(max_rows=50)}\n\n'
            self.memory.raw_text = all_text
            self.memory.structured_data = summary
            result['flags'].append(f'✅ Parsed {result["pages"]} sheet(s)')
        except Exception as e:
            result['flags'].append(f'❌ Excel extraction error: {e}')
        return result

    def _extract_csv(self, fp):
        result = {'method': 'Pandas (CSV)', 'flags': [], 'pages': 1}
        try:
            df = pd.read_csv(fp)
            self.memory.raw_text = df.to_string(max_rows=100)
            self.memory.structured_data = {
                'rows': len(df), 'cols': len(df.columns),
                'columns': list(df.columns),
                'numeric_cols': list(df.select_dtypes(include='number').columns),
                'null_pct': round(df.isnull().mean().mean() * 100, 1),
                'sample': df.head(3).to_dict(orient='records')
            }
            result['flags'].append(f'✅ Parsed CSV: {len(df)} rows × {len(df.columns)} cols')
        except Exception as e:
            result['flags'].append(f'❌ CSV extraction error: {e}')
        return result

    def _chunk_text(self, chunk_size=500, overlap=50):
        """Split text into overlapping chunks for RAG."""
        text = self.memory.raw_text
        if not text:
            return
        words = text.split()
        chunks = []
        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)
        self.memory.chunks = chunks


print('✅ Extractor Agent ready!')

## 🔍 Step 6 — Verifier Agent (All 5 Fraud Detection Layers)

In [ ]:
class VerifierAgent(BaseAgent):
    """
    Agent 2: Runs all 5 fraud detection layers.
    Metadata · ELA · Digital Signature · Text Consistency · QR Code
    """

    def run(self):
        self.log('Starting verification across 5 layers...')
        fp = self.memory.file_path

        layers = [
            self._metadata_analysis(fp),
            self._ela_analysis(fp),
            self._signature_check(fp),
            self._text_consistency(fp),
            self._qr_verification(fp),
        ]

        # Weighted fusion
        weights = [0.20, 0.25, 0.30, 0.15, 0.10]
        final_score = sum(l['score'] * w for l, w in zip(layers, weights))
        final_score = round(final_score, 1)

        if final_score >= 80:
            verdict, risk = '✅ Likely authentic', 'LOW'
        elif final_score >= 55:
            verdict, risk = '⚠️ Suspicious — review needed', 'MEDIUM'
        else:
            verdict, risk = '🚨 High risk of tampering', 'HIGH'

        result = {
            'layers': layers,
            'final_score': final_score,
            'verdict': verdict,
            'risk_level': risk,
            'ela_image': next((l.get('ela_image') for l in layers if l.get('ela_image')), None)
        }
        self.memory.verification_result = result
        self.log(f'Verification done. Score: {final_score}/100 — {verdict}')
        return result

    def _metadata_analysis(self, fp):
        r = {'layer': 'Metadata', 'score': 100, 'flags': [], 'details': {}}
        if not fp.lower().endswith('.pdf'):
            r['score'] = 55
            r['flags'].append('⚠️ Metadata analysis limited for non-PDF files')
            return r
        try:
            doc = fitz.open(fp)
            meta = doc.metadata
            r['details'] = {
                'author':   meta.get('author', 'N/A'),
                'creator':  meta.get('creator', 'N/A'),
                'producer': meta.get('producer', 'N/A'),
                'created':  meta.get('creationDate', 'N/A'),
                'modified': meta.get('modDate', 'N/A'),
            }
            bad_tools = ['photoshop','illustrator','gimp','foxit','nitro','pdf editor','smallpdf','ilovepdf','sejda']
            all_meta = (meta.get('creator','') + meta.get('producer','')).lower()
            if any(t in all_meta for t in bad_tools):
                r['flags'].append(f'🚨 Editing tool detected in metadata')
                r['score'] -= 45
            created  = meta.get('creationDate', '')
            modified = meta.get('modDate', '')
            if created and modified and created != modified:
                r['flags'].append('⚠️ Document was modified after creation')
                r['score'] -= 25
            gov_kw = ['nic','india','government','ministry','digilocker','uidai','nsdl']
            if any(k in ' '.join(meta.values()).lower() for k in gov_kw):
                r['flags'].append('✅ Government issuer found in metadata')
            else:
                r['flags'].append('⚠️ No government issuer in metadata')
                r['score'] -= 10
            raw = open(fp, 'rb').read().decode('latin-1', errors='ignore')
            prev_count = raw.count('/Prev')
            if prev_count > 1:
                r['flags'].append(f'⚠️ {prev_count} incremental saves — possible hidden edits')
                r['score'] -= 15
            doc.close()
        except Exception as e:
            r['flags'].append(f'❌ Metadata error: {e}')
            r['score'] = 40
        r['score'] = max(0, r['score'])
        return r

    def _ela_analysis(self, fp, quality=75):
        r = {'layer': 'ELA', 'score': 100, 'flags': [], 'details': {}, 'ela_image': None}
        try:
            if fp.lower().endswith('.pdf'):
                doc = fitz.open(fp)
                pix = doc[0].get_pixmap(matrix=fitz.Matrix(2,2))
                original = Image.open(io.BytesIO(pix.tobytes('png'))).convert('RGB')
                doc.close()
            else:
                original = Image.open(fp).convert('RGB')
            original = original.resize((800, 1000), Image.LANCZOS)
            buf = io.BytesIO()
            original.save(buf, 'JPEG', quality=quality)
            buf.seek(0)
            compressed = Image.open(buf).convert('RGB')
            diff = np.abs(np.array(original, dtype=np.float32) - np.array(compressed, dtype=np.float32))
            r['ela_image'] = Image.fromarray(np.clip(diff * 10, 0, 255).astype(np.uint8))
            gray = np.mean(diff, axis=2)
            hotspot_pct = float(np.sum(gray > np.mean(gray) + 2.5*np.std(gray)) / gray.size * 100)
            r['details'] = {'hotspot_pct': round(hotspot_pct, 2), 'mean_error': round(float(np.mean(diff)), 3)}
            if hotspot_pct > 5:
                r['flags'].append(f'🚨 High tampering signal: {hotspot_pct:.1f}% hotspot pixels')
                r['score'] -= 50
            elif hotspot_pct > 2:
                r['flags'].append(f'⚠️ Moderate signal: {hotspot_pct:.1f}% hotspot pixels')
                r['score'] -= 25
            else:
                r['flags'].append(f'✅ Low ELA variance ({hotspot_pct:.1f}%) — looks clean')
        except Exception as e:
            r['flags'].append(f'❌ ELA error: {e}')
            r['score'] = 50
        r['score'] = max(0, r['score'])
        return r

    def _signature_check(self, fp):
        r = {'layer': 'Digital Signature', 'score': 65, 'flags': [], 'details': {}}
        if not fp.lower().endswith('.pdf'):
            r['flags'].append('⚠️ Signature check requires PDF')
            return r
        try:
            raw = open(fp, 'rb').read().decode('latin-1', errors='ignore')
            has_byte_range = '/ByteRange' in raw
            has_pkcs       = '/PKCS7' in raw or 'adbe.pkcs7' in raw.lower()
            has_sig_dict   = '/Sig' in raw
            r['details'] = {'byte_range': has_byte_range, 'pkcs7': has_pkcs, 'sig_dict': has_sig_dict}
            if has_pkcs and has_byte_range:
                r['flags'].append('✅ Full PKCS7 digital certificate detected')
                r['score'] = 95
            elif has_sig_dict:
                r['flags'].append('✅ Signature dictionary found')
                r['score'] = 80
            else:
                r['flags'].append('⚠️ No digital signature — may be unsigned or stripped')
                r['score'] = 50
        except Exception as e:
            r['flags'].append(f'❌ Signature check error: {e}')
            r['score'] = 40
        return r

    def _text_consistency(self, fp):
        r = {'layer': 'Text Consistency', 'score': 100, 'flags': [], 'details': {}}
        text = self.memory.raw_text.lower()
        gov_kw = ['government of india','ministry','digilocker','aadhaar','unique identification',
                  'income tax','cbse','icse','driving licence','board of','university']
        found = [k for k in gov_kw if k in text]
        r['details']['gov_keywords'] = found
        if found:
            r['flags'].append(f'✅ Gov keywords: {", ".join(found[:3])}')
        else:
            r['flags'].append('⚠️ No government keywords detected')
            r['score'] -= 15
        dates = re.findall(r'\b\d{2}[/\-]\d{2}[/\-]\d{4}\b', text)
        r['details']['dates_found'] = dates[:5]
        if dates:
            r['flags'].append(f'✅ Dates found: {", ".join(dates[:2])}')
        if len(text.strip()) < 50:
            r['flags'].append('⚠️ Very little text — possible image-only or blank doc')
            r['score'] -= 20
        aadhaar = re.findall(r'\b\d{4}\s?\d{4}\s?\d{4}\b', text)
        pan     = re.findall(r'\b[a-z]{5}\d{4}[a-z]\b', text)
        if aadhaar: r['flags'].append('✅ Aadhaar number pattern found')
        if pan:     r['flags'].append('✅ PAN number pattern found')
        r['score'] = max(0, r['score'])
        return r

    def _qr_verification(self, fp):
        r = {'layer': 'QR Code', 'score': 60, 'flags': [], 'details': {}}
        try:
            if fp.lower().endswith('.pdf'):
                doc = fitz.open(fp)
                pix = doc[0].get_pixmap(matrix=fitz.Matrix(3,3))
                img = Image.open(io.BytesIO(pix.tobytes('png')))
                doc.close()
            else:
                img = Image.open(fp)
            img_cv = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
            decoded = qr_decode(img_cv)
            if decoded:
                data = decoded[0].data.decode('utf-8', errors='ignore')
                r['details']['qr_data'] = data[:200]
                gov_domains = ['digilocker.gov.in','uidai.gov.in','cbse.gov.in','.gov.in']
                if any(d in data.lower() for d in gov_domains):
                    r['flags'].append('✅ QR points to official government domain')
                    r['score'] = 95
                else:
                    r['flags'].append(f'⚠️ QR found but not a .gov.in domain')
                    r['score'] = 65
            else:
                r['flags'].append('⚠️ No QR code detected')
                r['score'] = 55
        except Exception as e:
            r['flags'].append(f'❌ QR error: {e}')
            r['score'] = 50
        return r


print('✅ Verifier Agent ready!')

## 🧠 Step 7 — Analyst Agent (AI Summary + RAG + Q&A)

In [ ]:
class AnalystAgent(BaseAgent):
    """
    Agent 3: AI-powered analysis using Claude API.
    - Summary of document content
    - Entity extraction (names, dates, amounts, IDs)
    - Key insights and risk flags
    - RAG setup for Q&A
    - Authenticity analysis using Claude's reasoning
    """

    def run(self):
        self.log('Starting AI analysis...')
        result = {
            'summary': '',
            'entities': {},
            'insights': [],
            'ai_authenticity_view': '',
            'rag_ready': False,
            'flags': []
        }

        # Build RAG index regardless of Claude availability
        if RAG_AVAILABLE and self.memory.chunks:
            self._build_rag_index()
            result['rag_ready'] = True
            result['flags'].append(f'✅ RAG index built: {len(self.memory.chunks)} chunks')

        if CLAUDE_AVAILABLE:
            result['summary']              = self._get_summary()
            result['entities']             = self._extract_entities()
            result['insights']             = self._get_insights()
            result['ai_authenticity_view'] = self._ai_authenticity_check()
        else:
            # Rule-based fallback when offline
            result['summary']   = self._rule_based_summary()
            result['entities']  = self._rule_based_entities()
            result['insights']  = self._rule_based_insights()
            result['ai_authenticity_view'] = 'AI analysis offline — using rule-based detection only'
            result['flags'].append('ℹ️ Running in offline mode — Claude API not available')

        self.memory.analysis_result = result
        self.log('AI analysis complete')
        return result

    def _build_rag_index(self):
        """Build FAISS vector index from document chunks."""
        try:
            embeddings = embedding_model.encode(self.memory.chunks, show_progress_bar=False)
            embeddings = np.array(embeddings, dtype=np.float32)
            faiss.normalize_L2(embeddings)
            index = faiss.IndexFlatIP(embeddings.shape[1])
            index.add(embeddings)
            self.memory.embeddings  = embeddings
            self.memory.faiss_index = index
            self.log(f'FAISS index built: {index.ntotal} vectors')
        except Exception as e:
            self.log(f'RAG index error: {e}')

    def _retrieve_chunks(self, query: str, top_k: int = 3) -> str:
        """Retrieve most relevant chunks for a query."""
        if not self.memory.faiss_index or not RAG_AVAILABLE:
            return self.memory.raw_text[:2000]
        try:
            q_emb = embedding_model.encode([query], show_progress_bar=False)
            q_emb = np.array(q_emb, dtype=np.float32)
            faiss.normalize_L2(q_emb)
            _, indices = self.memory.faiss_index.search(q_emb, top_k)
            return '\n\n'.join(self.memory.chunks[i] for i in indices[0] if i < len(self.memory.chunks))
        except Exception:
            return self.memory.raw_text[:2000]

    def _call_claude(self, system_prompt: str, user_prompt: str) -> str:
        """Call Claude API with retry."""
        try:
            response = claude_client.messages.create(
                model='claude-sonnet-4-20250514',
                max_tokens=1000,
                system=system_prompt,
                messages=[{'role': 'user', 'content': user_prompt}]
            )
            return response.content[0].text
        except Exception as e:
            return f'API error: {e}'

    def _get_summary(self) -> str:
        context = self._retrieve_chunks('document summary overview purpose')
        return self._call_claude(
            'You are a document analysis assistant. Be concise and clear.',
            f'Summarize this document in 3-4 sentences. What type of document is it? What is its main purpose?\n\nDocument:\n{context[:2000]}'
        )

    def _extract_entities(self) -> dict:
        context = self._retrieve_chunks('name date amount ID number address')
        response = self._call_claude(
            'You are an entity extraction assistant. Return ONLY valid JSON, no explanation.',
            f'''Extract all entities from this document. Return JSON with these keys:
{{"names": [], "dates": [], "amounts": [], "ids": [], "organizations": [], "addresses": []}}

Document:\n{context[:2000]}'''
        )
        try:
            clean = re.sub(r'```json|```', '', response).strip()
            return json.loads(clean)
        except Exception:
            return {'raw_response': response[:500]}

    def _get_insights(self) -> list:
        context = self._retrieve_chunks('key information risk important')
        ver = self.memory.verification_result
        score_context = f'Fraud detection score: {ver.get("final_score", "N/A")}/100. Verdict: {ver.get("verdict", "N/A")}'
        response = self._call_claude(
            'You are a document intelligence analyst. Return insights as a JSON array of strings.',
            f'''Based on the document and verification results, list 4-5 key insights.
{score_context}
Return ONLY a JSON array like: ["insight 1", "insight 2"]

Document:\n{context[:1500]}'''
        )
        try:
            clean = re.sub(r'```json|```', '', response).strip()
            result = json.loads(clean)
            return result if isinstance(result, list) else [response]
        except Exception:
            return [response[:300]]

    def _ai_authenticity_check(self) -> str:
        ver = self.memory.verification_result
        layer_summary = '\n'.join(
            f'- {l["layer"]}: {l["score"]}/100 — {" | ".join(l["flags"][:2])}'
            for l in ver.get('layers', [])
        )
        return self._call_claude(
            'You are a document forensics expert. Be direct and analytical.',
            f'''Based on these fraud detection results, give a 2-3 sentence expert opinion on document authenticity:

Overall Score: {ver.get("final_score")}/100
Verdict: {ver.get("verdict")}

Layer Results:
{layer_summary}

Document Text Preview:
{self.memory.raw_text[:800]}'''
        )

    # ── Rule-based fallbacks (no API needed) ──
    def _rule_based_summary(self) -> str:
        text = self.memory.raw_text[:500]
        ft   = self.memory.file_type.upper()
        return f'[{ft} Document] Extracted {len(self.memory.raw_text)} characters. Preview: {text[:200]}...'

    def _rule_based_entities(self) -> dict:
        text = self.memory.raw_text
        return {
            'dates':    re.findall(r'\b\d{2}[/\-]\d{2}[/\-]\d{4}\b', text)[:5],
            'aadhaar':  re.findall(r'\b\d{4}\s?\d{4}\s?\d{4}\b', text)[:3],
            'pan':      re.findall(r'\b[A-Z]{5}\d{4}[A-Z]\b', text)[:3],
            'amounts':  re.findall(r'(?:rs\.?|₹|inr)\s?[\d,]+', text.lower())[:5],
            'emails':   re.findall(r'[\w.]+@[\w.]+\.[a-z]{2,}', text)[:3],
            'phones':   re.findall(r'\b[6-9]\d{9}\b', text)[:3]
        }

    def _rule_based_insights(self) -> list:
        insights = []
        ver   = self.memory.verification_result
        score = ver.get('final_score', 0)
        insights.append(f'Document authenticity score: {score}/100')
        insights.append(f'Risk level: {ver.get("risk_level", "UNKNOWN")}')
        if score < 55:
            insights.append('Multiple forensic layers flagged anomalies — manual expert review strongly recommended')
        text = self.memory.raw_text.lower()
        if 'aadhaar' in text or 'unique identification' in text:
            insights.append('Document appears to be Aadhaar-related identity document')
        if 'income' in text or 'salary' in text:
            insights.append('Document appears to be financial/income related')
        return insights


print('✅ Analyst Agent ready!')

## 📊 Step 8 — Reporter Agent + Visualization

In [ ]:
class ReporterAgent(BaseAgent):
    """
    Agent 4: Assembles everything into a final structured report
    and generates visual dashboard.
    """

    def run(self):
        self.log('Building final report...')
        ext  = self.memory.extraction_result
        ver  = self.memory.verification_result
        anal = self.memory.analysis_result

        report = {
            'timestamp':       datetime.now().isoformat(),
            'file':            os.path.basename(self.memory.file_path),
            'file_type':       self.memory.file_type,
            'authenticity': {
                'score':       ver.get('final_score'),
                'verdict':     ver.get('verdict'),
                'risk_level':  ver.get('risk_level'),
                'layer_scores': {l['layer']: l['score'] for l in ver.get('layers', [])}
            },
            'extraction': {
                'method':      ext.get('method'),
                'pages':       ext.get('pages'),
                'text_length': ext.get('text_length'),
                'chunks':      ext.get('chunk_count')
            },
            'ai_analysis': {
                'summary':     anal.get('summary'),
                'entities':    anal.get('entities'),
                'insights':    anal.get('insights'),
                'expert_view': anal.get('ai_authenticity_view')
            },
            'all_flags': [
                flag
                for source in [ext, *ver.get('layers', []), anal]
                for flag in source.get('flags', [])
            ]
        }
        self.memory.final_report = report

        text_report = self._build_text_report(report, ver)
        viz_path    = self._build_visualization(ver, anal)

        self.log('Report complete!')
        return text_report, viz_path, report

    def _build_text_report(self, report, ver) -> str:
        lines = []
        lines.append('=' * 65)
        lines.append('   AI DOCUMENT INTELLIGENCE REPORT')
        lines.append('=' * 65)
        lines.append(f'  File      : {report["file"]}')
        lines.append(f'  Type      : {report["file_type"].upper()}')
        lines.append(f'  Analysed  : {report["timestamp"][:19]}')
        lines.append('')
        auth = report['authenticity']
        lines.append(f'  SCORE     : {auth["score"]} / 100')
        lines.append(f'  VERDICT   : {auth["verdict"]}')
        lines.append(f'  RISK      : {auth["risk_level"]}')
        lines.append('=' * 65)
        lines.append('')
        lines.append('  LAYER-BY-LAYER SCORES')
        lines.append('  ' + '─' * 55)
        for layer in ver.get('layers', []):
            bar = '█' * int(layer['score']//10) + '░' * (10 - int(layer['score']//10))
            lines.append(f'  {bar} {layer["score"]:3.0f}/100  {layer["layer"]}')
            for flag in layer['flags']:
                lines.append(f'          {flag}')
        lines.append('')
        lines.append('  AI SUMMARY')
        lines.append('  ' + '─' * 55)
        lines.append(f'  {report["ai_analysis"]["summary"]}')
        lines.append('')
        lines.append('  EXPERT AUTHENTICITY VIEW')
        lines.append('  ' + '─' * 55)
        lines.append(f'  {report["ai_analysis"]["expert_view"]}')
        lines.append('')
        lines.append('  KEY INSIGHTS')
        lines.append('  ' + '─' * 55)
        for ins in report['ai_analysis'].get('insights', []):
            lines.append(f'  • {ins}')
        lines.append('')
        lines.append('  EXTRACTED ENTITIES')
        lines.append('  ' + '─' * 55)
        for k, v in (report['ai_analysis'].get('entities') or {}).items():
            if v:
                lines.append(f'  {k:15}: {v}')
        lines.append('')
        lines.append('  AGENT ACTIVITY LOG')
        lines.append('  ' + '─' * 55)
        for entry in self.memory.agent_log[-15:]:
            lines.append(f'  {entry}')
        lines.append('')
        lines.append('  ⚠️  Prototype for educational/demonstration purposes.')
        lines.append('  Always verify documents through official channels.')
        lines.append('=' * 65)
        return '\n'.join(lines)

    def _build_visualization(self, ver, anal) -> str:
        ela_image = ver.get('ela_image')
        fig = plt.figure(figsize=(16, 9))
        fig.patch.set_facecolor('#0f0f0f')

        # Gauge
        ax1 = fig.add_subplot(2, 3, 1)
        ax1.set_facecolor('#1a1a1a')
        score = ver.get('final_score', 50)
        color = '#2ecc71' if score >= 80 else '#f39c12' if score >= 55 else '#e74c3c'
        theta = np.linspace(np.pi, 0, 200)
        ax1.plot(np.cos(theta), np.sin(theta), color='#333', linewidth=18, solid_capstyle='round')
        theta_s = np.linspace(np.pi, np.pi - (score/100)*np.pi, 200)
        ax1.plot(np.cos(theta_s), np.sin(theta_s), color=color, linewidth=18, solid_capstyle='round')
        ax1.text(0, 0.1, f'{score:.0f}', ha='center', va='center', fontsize=38, color='white', fontweight='bold')
        ax1.text(0, -0.3, '/100', ha='center', va='center', fontsize=14, color='#aaa')
        verdict_short = ver.get('verdict', '')[:20]
        ax1.text(0, -0.6, verdict_short, ha='center', va='center', fontsize=9, color=color)
        ax1.set_xlim(-1.3, 1.3); ax1.set_ylim(-0.8, 1.1)
        ax1.set_title('Authenticity Score', color='white', fontsize=11, pad=8); ax1.axis('off')

        # Layer bars
        ax2 = fig.add_subplot(2, 3, 2)
        ax2.set_facecolor('#1a1a1a')
        layers  = [l['layer'] for l in ver.get('layers', [])]
        scores  = [l['score'] for l in ver.get('layers', [])]
        colors  = ['#2ecc71' if s >= 80 else '#f39c12' if s >= 55 else '#e74c3c' for s in scores]
        bars    = ax2.barh(layers, scores, color=colors, height=0.5, edgecolor='none')
        ax2.set_xlim(0, 100)
        ax2.axvline(x=80, color='#2ecc71', linestyle='--', alpha=0.4, linewidth=1)
        ax2.axvline(x=55, color='#f39c12', linestyle='--', alpha=0.4, linewidth=1)
        for bar, s in zip(bars, scores):
            ax2.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{s}', va='center', color='white', fontsize=9)
        ax2.tick_params(colors='white', labelsize=8)
        ax2.set_title('Layer Scores', color='white', fontsize=11, pad=8)
        for sp in ax2.spines.values(): sp.set_visible(False)

        # ELA
        ax3 = fig.add_subplot(2, 3, 3)
        if ela_image:
            ax3.imshow(ela_image, cmap='hot')
            ax3.set_title('ELA Heatmap\n(Bright = Suspicious)', color='white', fontsize=11, pad=8)
        else:
            ax3.text(0.5, 0.5, 'ELA N/A', ha='center', va='center', color='white', fontsize=12)
            ax3.set_facecolor('#1a1a1a')
            ax3.set_title('ELA Heatmap', color='white', fontsize=11, pad=8)
        ax3.axis('off')

        # AI Summary
        ax4 = fig.add_subplot(2, 3, 4)
        ax4.set_facecolor('#1a1a1a'); ax4.axis('off')
        summary = (anal.get('summary') or 'N/A')[:400]
        ax4.text(0.03, 0.95, 'AI Summary', fontsize=10, color='white', fontweight='bold', va='top', transform=ax4.transAxes)
        ax4.text(0.03, 0.82, summary, fontsize=8, color='#cccccc', va='top', transform=ax4.transAxes, wrap=True)
        ax4.set_title('', color='white')

        # Entities
        ax5 = fig.add_subplot(2, 3, 5)
        ax5.set_facecolor('#1a1a1a'); ax5.axis('off')
        entities = anal.get('entities') or {}
        ent_text = '\n'.join(f'{k}: {v}' for k, v in entities.items() if v)[:400]
        ax5.text(0.03, 0.95, 'Extracted Entities', fontsize=10, color='white', fontweight='bold', va='top', transform=ax5.transAxes)
        ax5.text(0.03, 0.82, ent_text or 'No entities extracted', fontsize=8, color='#cccccc', va='top', transform=ax5.transAxes, family='monospace')

        # Insights
        ax6 = fig.add_subplot(2, 3, 6)
        ax6.set_facecolor('#1a1a1a'); ax6.axis('off')
        insights = anal.get('insights') or []
        ins_text = '\n'.join(f'• {i}' for i in insights[:5])
        ax6.text(0.03, 0.95, 'Key Insights', fontsize=10, color='white', fontweight='bold', va='top', transform=ax6.transAxes)
        ax6.text(0.03, 0.82, ins_text or 'No insights', fontsize=8, color='#cccccc', va='top', transform=ax6.transAxes)

        plt.suptitle('AI Document Intelligence System — Analysis Dashboard', color='white', fontsize=13, y=0.99)
        plt.tight_layout(rect=[0, 0, 1, 0.98])
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        plt.savefig(tmp.name, dpi=120, bbox_inches='tight', facecolor='#0f0f0f')
        plt.close()
        return tmp.name


print('✅ Reporter Agent ready!')

## 🎯 Step 9 — Orchestrator + RAG Q&A

In [ ]:
import os

def run_full_pipeline(uploaded_file):
    """Orchestrator: coordinates all 4 agents in sequence."""
    if uploaded_file is None:
        return 'Please upload a document.', None, '{}'

    fp = uploaded_file if isinstance(uploaded_file, str) else uploaded_file.name

    print('\n' + '='*50)
    print(f'  ORCHESTRATOR: Starting pipeline')
    print(f'  File: {os.path.basename(fp)}')
    print('='*50)

    # Shared memory for all agents
    memory = AgentMemory(file_path=fp)

    # Run agents in order
    ExtractorAgent('EXTRACTOR', memory).run()
    VerifierAgent('VERIFIER',   memory).run()
    AnalystAgent('ANALYST',     memory).run()
    text_report, viz_path, json_report = ReporterAgent('REPORTER', memory).run()

    json_str = json.dumps(json_report, indent=2, default=str)
    print('\n✅ ORCHESTRATOR: Pipeline complete!')
    return text_report, viz_path, json_str


def ask_document(question, state_json):
    """RAG-powered Q&A on the last analysed document."""
    if not state_json or state_json == '{}':
        return 'Please analyse a document first.'
    if not CLAUDE_AVAILABLE:
        return 'Claude API not available. Please add your API key in Step 2.'
    if not RAG_AVAILABLE:
        return 'RAG not available — sentence-transformers or FAISS not installed.'

    # Retrieve relevant chunks from last analysis
    try:
        state = json.loads(state_json)
        # We store document text in the state for Q&A
        doc_text = state.get('_doc_text', '')[:3000]
        response = claude_client.messages.create(
            model='claude-sonnet-4-20250514',
            max_tokens=500,
            system='You are a helpful document Q&A assistant. Answer questions based only on the provided document text.',
            messages=[{
                'role': 'user',
                'content': f'Document:\n{doc_text}\n\nQuestion: {question}'
            }]
        )
        return response.content[0].text
    except Exception as e:
        return f'Q&A error: {e}'


print('✅ Orchestrator ready!')

## 🖥️ Step 10 — Full Gradio UI with Q&A

In [ ]:
import gradio as gr

with gr.Blocks(title='AI Document Intelligence System', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🤖 AI Document Intelligence & Agent System
    ### HackBaroda 2026 — Full Multi-Agent System
    Upload any document → 4 AI agents analyse it → get authenticity verdict + AI summary + insights
    """)

    report_state = gr.State('{}')

    with gr.Tabs():

        # ── TAB 1: ANALYSIS
        with gr.Tab('Document Analysis'):
            with gr.Row():
                with gr.Column(scale=1):
                    file_input = gr.File(
                        label='Upload Document',
                        file_types=['.pdf','.png','.jpg','.jpeg','.xlsx','.csv'],
                        type='filepath'
                    )
                    analyse_btn = gr.Button('Analyse Document', variant='primary', size='lg')
                    gr.Markdown("""
                    **Agents running:**
                    - Extractor → reads all content
                    - Verifier → 5-layer fraud check
                    - Analyst → AI summary + entities
                    - Reporter → final report + dashboard
                    """)
                with gr.Column(scale=2):
                    report_out = gr.Textbox(label='Full Report', lines=30, show_copy_button=True)

            viz_out  = gr.Image(label='Visual Dashboard', type='filepath')
            json_out = gr.Code(label='Structured JSON Output', language='json')

            analyse_btn.click(
                fn=run_full_pipeline,
                inputs=[file_input],
                outputs=[report_out, viz_out, json_out]
            )

        # ── TAB 2: Q&A
        with gr.Tab('Ask the Document (Q&A)'):
            gr.Markdown("""
            ### Ask any question about the last analysed document
            *Requires Claude API key + document to be analysed first*
            """)
            question_input = gr.Textbox(
                label='Your Question',
                placeholder='e.g. What is the name on this document? What is the issue date?',
                lines=2
            )
            ask_btn  = gr.Button('Ask', variant='primary')
            qa_out   = gr.Textbox(label='Answer', lines=6)
            ask_btn.click(fn=ask_document, inputs=[question_input, json_out], outputs=[qa_out])

        # ── TAB 3: AGENT LOG
        with gr.Tab('How to Use'):
            gr.Markdown("""
            ## Quick Guide

            ### Step 1 — Upload any document
            Supports: PDF, PNG, JPG, JPEG, Excel (.xlsx), CSV

            ### Step 2 — Click Analyse Document
            All 4 agents run automatically:
            1. **Extractor** reads and cleans content
            2. **Verifier** runs 5-layer fraud check
            3. **Analyst** generates AI summary and insights
            4. **Reporter** builds final dashboard

            ### Step 3 — Read the results
            - **Score 80-100** → Likely authentic
            - **Score 55-79** → Suspicious, review manually
            - **Score 0-54**  → High risk of tampering

            ### Step 4 — Ask questions (Q&A tab)
            Requires Claude API key. Ask anything about the document.

            ### To test tampering detection:
            1. Upload an original PDF → note the score
            2. Edit it in Smallpdf.com → re-upload
            3. Score should drop significantly

            ### Datasets to test with:
            - Your own DigiLocker PDF (best real test)
            - CASIA TIDE v2 dataset (tampered images)
            - Any Excel CSV from Kaggle
            """)

demo.launch(share=True, debug=True)